In [26]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

rd.open_session()

<refinitiv.data.session.Definition object at 0x141205f10 {name='workspace'}>

In [27]:
universe = ['NESTE.HE']

fields = [
    "TR.PriceClose.date",
    "TR.PriceClose", # "dirty price" without dividends
    "TR.TotalReturn1D", # dividends included
]

params = {
    "SDate": "2024-01-01",
    "EDate": "2026-03-05",
    "Frq": "D",
    "Curn": "EUR"
}

df = rd.get_data(universe=universe, 
                 fields=fields, 
                 parameters=params)


df.head()


,Instrument,Date,Price Close,Daily Total Return
0,NESTE.HE,2024-01-02,32.48,0.838249
1,NESTE.HE,2024-01-03,31.8,-2.093596
2,NESTE.HE,2024-01-04,32.27,1.477987
3,NESTE.HE,2024-01-05,32.4,0.402851
4,NESTE.HE,2024-01-08,32.27,-0.401235


In [28]:
df['Daily Total Return Manual'] = df['Price Close'].pct_change()

#### Tätä käytetään betan laskennassa

In [29]:
df['Daily Total Return'] = df['Daily Total Return'] / 100

In [30]:
df

,Instrument,Date,Price Close,Daily Total Return,Daily Total Return Manual
0,NESTE.HE,2024-01-02,32.48,0.008382,<NA>
1,NESTE.HE,2024-01-03,31.8,-0.020936,-0.020936
2,NESTE.HE,2024-01-04,32.27,0.01478,0.01478
3,NESTE.HE,2024-01-05,32.4,0.004029,0.004029
4,NESTE.HE,2024-01-08,32.27,-0.004012,-0.004012
...,...,...,...,...,...
540,NESTE.HE,2026-02-27,21.18,-0.005634,-0.005634
541,NESTE.HE,2026-03-02,22.65,0.069405,0.069405
542,NESTE.HE,2026-03-03,22.71,0.002649,0.002649
543,NESTE.HE,2026-03-04,22.53,-0.007926,-0.007926


In [31]:
df['Log Daily Total Return'] = np.log(1 + df['Daily Total Return'])

In [ ]:
df['Log Daily Total Return Manual'] = np.log(1 + df['Daily Total Return Manual'])

In [41]:
df 

,Instrument,Date,Price Close,Daily Total Return,Daily Total Return Manual,Log Daily Total Return,Log Daily Total Return Manual
0,NESTE.HE,2024-01-02,32.48,0.008382,<NA>,0.008348,<NA>
1,NESTE.HE,2024-01-03,31.8,-0.020936,-0.020936,-0.021158,-0.021158
2,NESTE.HE,2024-01-04,32.27,0.01478,0.01478,0.014672,0.014672
3,NESTE.HE,2024-01-05,32.4,0.004029,0.004029,0.00402,0.00402
4,NESTE.HE,2024-01-08,32.27,-0.004012,-0.004012,-0.00402,-0.00402
...,...,...,...,...,...,...,...
540,NESTE.HE,2026-02-27,21.18,-0.005634,-0.005634,-0.00565,-0.00565
541,NESTE.HE,2026-03-02,22.65,0.069405,0.069405,0.067103,0.067103
542,NESTE.HE,2026-03-03,22.71,0.002649,0.002649,0.002646,0.002646
543,NESTE.HE,2026-03-04,22.53,-0.007926,-0.007926,-0.007958,-0.007958


In [ ]:
df.rename(columns={
    'Daily Total Return Manual': 'Daily Return excl. dividends',
    'Log Daily Total Return Manual': 'Log Daily Return excl. dividends',
}, inplace=True)


In [43]:
df

,Instrument,Date,Price Close,Daily Total Return,Daily Return excl. dividends,Log Daily Total Return,Log Daily Return excl. dividends
0,NESTE.HE,2024-01-02,32.48,0.008382,<NA>,0.008348,<NA>
1,NESTE.HE,2024-01-03,31.8,-0.020936,-0.020936,-0.021158,-0.021158
2,NESTE.HE,2024-01-04,32.27,0.01478,0.01478,0.014672,0.014672
3,NESTE.HE,2024-01-05,32.4,0.004029,0.004029,0.00402,0.00402
4,NESTE.HE,2024-01-08,32.27,-0.004012,-0.004012,-0.00402,-0.00402
...,...,...,...,...,...,...,...
540,NESTE.HE,2026-02-27,21.18,-0.005634,-0.005634,-0.00565,-0.00565
541,NESTE.HE,2026-03-02,22.65,0.069405,0.069405,0.067103,0.067103
542,NESTE.HE,2026-03-03,22.71,0.002649,0.002649,0.002646,0.002646
543,NESTE.HE,2026-03-04,22.53,-0.007926,-0.007926,-0.007958,-0.007958


In [39]:
universe = [".STOXX",
            ".STOXXR"
]

fields = [
    "TR.PriceClose.date",
    "TR.PriceClose",
]

params = {
    "SDate": "2024-01-01",
    "EDate": "2026-03-05",
    "Frq": "D",
    "Curn": "EUR"
}

df_index = rd.get_data(
    universe=universe,
    fields=fields,
    parameters=params
)

df_index.head()

,Instrument,Date,Price Close
0,.STOXX,2023-12-29,478.99
1,.STOXX,2024-01-02,478.51
2,.STOXX,2024-01-03,474.4
3,.STOXX,2024-01-04,477.68
4,.STOXX,2024-01-05,476.38


In [ ]:
# Varmista Date on datetime ilman kellonaikaa
df_index['Date'] = pd.to_datetime(df_index['Date']).dt.date

# Pudota mahdolliset duplikaatit
df_index = df_index.drop_duplicates(subset=['Date', 'Instrument'], keep='last')

df_pivot = df_index.pivot(index='Date', columns='Instrument', values='Price Close')
df_pivot

Instrument,.STOXX,.STOXXR
Date,,
2023-12-29,478.99,1136.87
2024-01-02,478.51,1135.88
2024-01-03,474.4,1126.12
2024-01-04,477.68,1133.94
2024-01-05,476.38,1130.86
...,...,...
2026-02-27,633.85,1588.37
2026-03-02,623.36,1562.76
2026-03-03,604.44,1514.68


In [37]:
# calculate the total return for the period for both indices
total_return_stoxx = (df_pivot['.STOXX'].iloc[-1] / df_pivot['.STOXX'].iloc[0]) - 1
total_return_stoxxr = (df_pivot['.STOXXR'].iloc[-1] / df_pivot['.STOXXR'].iloc[0]) - 1

In [38]:
total_return_stoxx, total_return_stoxxr

(np.float64(0.2627194722228021), np.float64(0.3334154300843546))

In [44]:
df_index

,Instrument,Date,Price Close
0,.STOXX,2023-12-29,478.99
1,.STOXX,2024-01-02,478.51
2,.STOXX,2024-01-03,474.4
3,.STOXX,2024-01-04,477.68
4,.STOXX,2024-01-05,476.38
...,...,...,...
1133,.STOXXR,2026-02-27,1588.37
1134,.STOXXR,2026-03-02,1562.76
1135,.STOXXR,2026-03-03,1514.68
1136,.STOXXR,2026-03-04,1535.4


In [45]:
df_index['Daily Return'] = df_index['Price Close'].pct_change()

In [46]:
df_index

,Instrument,Date,Price Close,Daily Return
0,.STOXX,2023-12-29,478.99,<NA>
1,.STOXX,2024-01-02,478.51,-0.001002
2,.STOXX,2024-01-03,474.4,-0.008589
3,.STOXX,2024-01-04,477.68,0.006914
4,.STOXX,2024-01-05,476.38,-0.002721
...,...,...,...,...
1133,.STOXXR,2026-02-27,1588.37,0.001097
1134,.STOXXR,2026-03-02,1562.76,-0.016123
1135,.STOXXR,2026-03-03,1514.68,-0.030766
1136,.STOXXR,2026-03-04,1535.4,0.013679


In [47]:
df_index['Daily Log Return'] = np.log(1 + df_index['Daily Return'])

In [48]:
df_index

,Instrument,Date,Price Close,Daily Return,Daily Log Return
0,.STOXX,2023-12-29,478.99,<NA>,<NA>
1,.STOXX,2024-01-02,478.51,-0.001002,-0.001003
2,.STOXX,2024-01-03,474.4,-0.008589,-0.008626
3,.STOXX,2024-01-04,477.68,0.006914,0.00689
4,.STOXX,2024-01-05,476.38,-0.002721,-0.002725
...,...,...,...,...,...
1133,.STOXXR,2026-02-27,1588.37,0.001097,0.001096
1134,.STOXXR,2026-03-02,1562.76,-0.016123,-0.016255
1135,.STOXXR,2026-03-03,1514.68,-0.030766,-0.031249
1136,.STOXXR,2026-03-04,1535.4,0.013679,0.013587


In [57]:
# pivotoi nyt toi taulukko mahd yksinkertaisesti    
df_pivot = df_index.pivot(index='Date', columns='Instrument', values=['Price Close', 'Daily Return', 'Daily Log Return'])

In [58]:
df_pivot

Price Close          Daily Return           Daily Log Return  \
Instrument      .STOXX  .STOXXR       .STOXX   .STOXXR           .STOXX   
Date                                                                      
2023-12-29      478.99  1136.87         <NA>  0.879652             <NA>   
2024-01-02      478.51  1135.88    -0.001002 -0.000871        -0.001003   
2024-01-03       474.4  1126.12    -0.008589 -0.008592        -0.008626   
2024-01-04      477.68  1133.94     0.006914  0.006944          0.00689   
2024-01-05      476.38  1130.86    -0.002721 -0.002716        -0.002725   
...                ...      ...          ...       ...              ...   
2026-02-27      633.85  1588.37     0.001058  0.001097         0.001058   
2026-03-02      623.36  1562.76     -0.01655 -0.016123        -0.016688   
2026-03-03      604.44  1514.68    -0.030352 -0.030766        -0.030822   
2026-03-04      612.71   1535.4     0.013682  0.013679         0.013589   
2026-03-05      604.83  1515.92    -0.012861 -0.012687        -0.012944   

                      
Instrument   .STOXXR  
Date                  
2023-12-29  0.631087  
2024-01-02 -0.000871  
2024-01-03  -0.00863  
2024-01-04   0.00692  
2024-01-05  -0.00272  
...              ...  
2026-02-27  0.001096  
2026-03-02 -0.016255  
2026-03-03 -0.031249  
2026-03-04  0.013587  
2026-03-05 -0.012768  

[559 rows x 6 columns]